# Solutions · 13 · Suspensions and emulsions

Worked solutions to the exercises of [notebook 13](../notebooks/13_suspensions_and_emulsions.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import datasets, suspensions as su
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
gb = pd.read_csv(datasets.path("glass_beads_suspension.csv"), comment="#")
phi, eta_r = gb.volume_fraction.to_numpy(), gb.relative_viscosity.to_numpy()
pm = su.fit_krieger_dougherty(phi, eta_r)["phi_max"]

## Exercise 1

What solids fraction gives a relative viscosity of 10 with the fitted $\phi_{max}$? And of 100?

In [3]:
for target in (10.0, 100.0):
    p_ = pm * (1 - target ** (-1 / (2.5 * pm)))
    print(f"relative viscosity {target:5.0f} at phi = {p_:.3f}  (check: {su.krieger_dougherty(p_, pm):.1f})")

relative viscosity    10 at phi = 0.477  (check: 10.0)
relative viscosity   100 at phi = 0.583  (check: 100.0)


Inverting Krieger-Dougherty: going from a tenfold to a hundredfold viscosity needs only about 0.1 more solids
(0.48 to 0.58) - the steep approach to maximum packing again.

## Exercise 2

At a fixed solids content of 0.55, by what factor does the viscosity fall if a broader size distribution raises $\phi_{max}$ from the fitted value to 0.70?

In [4]:
print(f"eta_r at phi = 0.55: phi_max {pm:.3f} -> {su.krieger_dougherty(0.55, pm):.1f};  phi_max 0.70 -> {su.krieger_dougherty(0.55, 0.70):.1f}"
      f"  (factor {su.krieger_dougherty(0.55, pm)/su.krieger_dougherty(0.55, 0.70):.1f})")

eta_r at phi = 0.55: phi_max 0.614 -> 32.4;  phi_max 0.70 -> 14.8  (factor 2.2)


A broader size distribution that raises the maximum packing fraction to 0.70 cuts the viscosity at the same
solids content by a factor of more than two - or allows considerably more solids at the same viscosity.

## Exercise 3

**Implement it yourself:** tabulate the Einstein, Batchelor and Krieger-Dougherty predictions at $\phi$ = 0.02, 0.05, 0.1 and 0.2 and compare them with the data. Where does each fail?

In [5]:
low = phi <= 0.2
table = pd.DataFrame({"phi": phi[low], "measured": eta_r[low], "Einstein": su.einstein(phi[low]),
                      "Batchelor": su.batchelor(phi[low]), "Krieger-Dougherty": su.krieger_dougherty(phi[low], pm)})
print(table.round(3).to_string(index=False))

 phi  measured  Einstein  Batchelor  Krieger-Dougherty
0.02     1.027     1.050      1.052              1.052
0.05     1.137     1.125      1.140              1.139
0.10     1.253     1.250      1.312              1.314
0.15     1.569     1.375      1.514              1.537
0.20     1.732     1.500      1.748              1.831


All three agree in the dilute limit, within the 3 % scatter of the data (at $\phi$ = 0.1 Einstein even happens to lie
closest). From $\phi$ = 0.15 Einstein falls clearly behind (it ignores particle interactions), Batchelor holds a little
further, and only Krieger-Dougherty follows the data up to high concentrations (notebook 13).